In [1]:
import pandas as pd 
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_csv('orders.csv')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,order_date_fixed
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016-11-08
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016-11-08
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016-06-12
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015-10-11
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015-10-11


In [3]:
print(df['order_date_fixed'].dtype)

str


In [4]:
df['order_date_fixed'] = pd.to_datetime(df['order_date_fixed'])

In [5]:
print(df['order_date_fixed'].dtype)

datetime64[us]


In [7]:
snapshot_date = df['order_date_fixed'].max() + pd.Timedelta(days = 1)
print(snapshot_date)

2017-12-31 00:00:00


In [8]:
rfm = df.groupby('Customer ID').agg(
    Recency = ('order_date_fixed', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Order ID', 'nunique'),
    Monetary = ('Profit','sum')
).reset_index()

print(rfm.head())

  Customer ID  Recency  Frequency   Monetary
0    AA-10375      539          2    63.5330
1    AA-10480      260          1     5.4432
2    AB-10060      486          1  1453.4346
3    AB-10165      261          2    69.8910
4    AC-10420     1268          1   -14.6632


In [9]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels = [5, 4, 3, 2, 1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method= 'first'), 5, labels = [1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels = [1, 2, 3, 4, 5])

print(rfm.head())

  Customer ID  Recency  Frequency   Monetary R_score F_score M_score
0    AA-10375      539          2    63.5330       3       4       4
1    AA-10480      260          1     5.4432       4       1       2
2    AB-10060      486          1  1453.4346       3       1       5
3    AB-10165      261          2    69.8910       4       4       4
4    AC-10420     1268          1   -14.6632       1       1       1


In [10]:
rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

def segment_customer(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r>= 4 and f>=4 and m>= 4:
        return 'Champions'
    elif r>= 4 and f>=3:
        return 'Loyal Customers'
    elif r >=4 and f<=2:
        return 'New/Promising'
    elif r<=2 and f<=2 and m<=2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['Segment'] = rfm.apply(segment_customer, axis = 1)
print(rfm['Segment'].value_counts())

Segment
Needs Attention    187
Loyal Customers     56
New/Promising       53
Champions           38
Lost                30
Name: count, dtype: int64


In [11]:
print(rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending = False))

Segment
Needs Attention    17651.1925
Champions           9392.0183
New/Promising       -687.2114
Loyal Customers    -1978.4903
Lost               -3067.5757
Name: Monetary, dtype: float64


In [12]:
print(rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending = False))

Segment
Champions          247.158376
Needs Attention     94.391404
New/Promising      -12.966253
Loyal Customers    -35.330184
Lost              -102.252523
Name: Monetary, dtype: float64


In [13]:
loyal_ids = rfm[rfm['Segment'] == 'Loyal Customers']['Customer ID']
loyal_orders = df[df['Customer ID'].isin(loyal_ids)]
print(loyal_orders.groupby('Category')['Profit'].mean().sort_values())

Category
Furniture         -83.240483
Office Supplies     5.779780
Technology         31.617514
Name: Profit, dtype: float64
